In [ ]:
!git clone https://github.com/SUFIYAN2004/quantize-as-a-tool
%cd quantize-as-a-tool
!pip install -q -U -r requirements.txt
%run quantize_service.py
server, key, url = start_service()
print(url)

In [ ]:
import requests, json
url = open("/content/public_url.txt").read().strip()
key = open("/content/api_key.txt").read().strip()

print("health  ->", requests.get(url + "/health", timeout=30).text)

r = requests.post(url + "/quantize",
                  json={"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "accuracy_budget_pct": 10.0},
                  headers={"X-API-Key": key}, timeout=900)
d = r.json()
print(r.status_code, d.get("status"), d.get("winner"), d.get("winner_stats"))
for a in d.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

In [ ]:
print(open("/content/public_url.txt").read())
print(open("/content/api_key.txt").read())

In [ ]:
!pip install -q "mcp[cli]"

In [ ]:
%%writefile /content/mcp_server.py
import os
import requests
from mcp.server.fastmcp import FastMCP

URL = os.environ.get("QUANTIZE_URL", "").rstrip("/")
KEY = os.environ.get("QUANTIZE_API_KEY", "")

mcp = FastMCP("quantize-as-a-tool")


def _slim(d: dict) -> dict:
    keep = ["status", "model_id", "winner", "winner_stats", "fp16_size_gb",
            "budget_pct", "artifact_sha256", "artifact_dir", "flags"]
    out = {k: d[k] for k in keep if k in d}
    out["attempts"] = [
        {k: a[k] for k in ("config", "ppl_increase_pct", "size_gb", "verdict")}
        for a in d.get("attempts", [])
    ]
    return out


@mcp.tool()
def quantize_model(model_id: str, accuracy_budget_pct: float = 10.0, bits: str = "auto") -> dict:
    """Compress a Hugging Face model (up to 3B params) to 4-bit or 8-bit.

    Returns the smallest format whose WikiText-2 perplexity increase stays within
    accuracy_budget_pct, plus the measured size, quality change and a SHA-256 of
    the artifact. bits is "auto", "4" or "8". Gated, remote-code, pickle-only or
    oversized models are refused with flags explaining why.
    """
    if not URL or not KEY:
        return {"status": "ERROR", "detail": "Set QUANTIZE_URL and QUANTIZE_API_KEY."}
    try:
        r = requests.post(
            f"{URL}/quantize",
            json={"model_id": model_id, "accuracy_budget_pct": accuracy_budget_pct, "bits": bits},
            headers={"X-API-Key": KEY},
            timeout=900,
        )
    except requests.RequestException as e:
        return {"status": "ERROR", "detail": f"Service unreachable: {type(e).__name__}"}
    if r.status_code == 401:
        return {"status": "UNAUTHORIZED", "detail": "Wrong API key."}
    if r.status_code != 200:
        return {"status": "ERROR", "detail": f"HTTP {r.status_code}"}
    return _slim(r.json())


@mcp.tool()
def service_health() -> dict:
    """Check whether the quantization service is reachable."""
    try:
        return requests.get(f"{URL}/health", timeout=20).json()
    except Exception as e:
        return {"ok": False, "detail": type(e).__name__}


if __name__ == "__main__":
    mcp.run()

In [ ]:
!pip install -q "mcp[cli]<2"

import sys
for m in [m for m in sys.modules if m == "mcp" or m.startswith("mcp.") or m == "mcp_server"]:
    del sys.modules[m]

from mcp.server.fastmcp import FastMCP
print("FastMCP import OK")

In [ ]:
import os, sys, json
os.environ["QUANTIZE_URL"] = open("/content/public_url.txt").read().strip()
os.environ["QUANTIZE_API_KEY"] = open("/content/api_key.txt").read().strip()
sys.path.insert(0, "/content")
import mcp_server

out = {"health": mcp_server.service_health(),
       "tinyllama": mcp_server.quantize_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0", 10.0),
       "qwen7b": mcp_server.quantize_model("Qwen/Qwen2.5-7B", 10.0)}
json.dump(out, open("/content/mcp_test.json", "w"), indent=2)
print(open("/content/mcp_test.json").read())

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import os, sys

params = StdioServerParameters(
    command=sys.executable, args=["/content/mcp_server.py"],
    env={**os.environ,
         "QUANTIZE_URL": open("/content/public_url.txt").read().strip(),
         "QUANTIZE_API_KEY": open("/content/api_key.txt").read().strip()})

with open("/content/mcp_err.log", "w") as errf:
    async with stdio_client(params, errlog=errf) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            tools = await s.list_tools()
            print("tools:", [t.name for t in tools.tools])
            res = await s.call_tool("quantize_model", {"model_id": "Qwen/Qwen2.5-7B"})
            print(res.content[0].text[:300])

In [ ]:
from google.colab import files
files.download("/content/mcp_server.py")

In [ ]:
import requests, json
key = open("/content/api_key.txt").read().strip()

r = requests.post("http://127.0.0.1:8000/quantize",
                  json={"model_id": "microsoft/phi-2", "accuracy_budget_pct": 10.0},
                  headers={"X-API-Key": key}, timeout=3000)
d = r.json()
json.dump(d, open("/content/api_result_phi2.json", "w"), indent=2)

print(r.status_code, d.get("status"), d.get("winner"), d.get("winner_stats"), d.get("flags"))
for a in d.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

In [ ]:
from google.colab import files
files.download("/content/quantize-as-a-tool/memory.json")
files.download("/content/api_result_phi2.json")

In [ ]:
import time, requests, json

# 1. patch quantize_service.py
p = "/content/quantize-as-a-tool/quantize_service.py"
src = open(p).read()
if "/jobs" not in src:
    new = '''
import uuid
JOBS = {}

def _run_job(job_id, req):
    JOBS[job_id]["status"] = "RUNNING"
    try:
        JOBS[job_id]["result"] = quantize_endpoint(req)
        JOBS[job_id]["status"] = "DONE"
    except Exception as e:
        JOBS[job_id]["status"] = "FAILED"
        JOBS[job_id]["error"] = type(e).__name__

@app.post("/jobs")
def submit_job(req: Req):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"status": "QUEUED", "model_id": req.model_id}
    threading.Thread(target=_run_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "QUEUED"}

@app.get("/jobs/{job_id}")
def job_status(job_id: str):
    j = JOBS.get(job_id)
    return j if j else JSONResponse({"status": "NOT_FOUND"}, status_code=404)

'''
    src = src.replace("def start_service(", new + "def start_service(", 1)
    src = src.replace('scope["path"] == "/quantize"',
                      '(scope["path"] == "/quantize" or scope["path"].startswith("/jobs"))', 1)
    open(p, "w").write(src)

# 2. restart the service with the patched file
try: server.should_exit = True
except Exception: pass
time.sleep(4)
get_ipython().run_line_magic("run", p)
server, key, url = start_service()
print("URL:", url)

# 3. submit a new model and poll
H = {"X-API-Key": key}
job = requests.post(url + "/jobs", headers=H, timeout=60,
                    json={"model_id": "Qwen/Qwen2.5-1.5B", "accuracy_budget_pct": 10.0}).json()
print("submitted:", job)

last = None
for _ in range(120):                      # up to ~20 minutes
    j = requests.get(f"{url}/jobs/{job['job_id']}", headers=H, timeout=60).json()
    if j["status"] != last:
        print(time.strftime("%H:%M:%S"), j["status"]); last = j["status"]
    if j["status"] in ("DONE", "FAILED"):
        break
    time.sleep(10)

json.dump(j, open("/content/job_result.json", "w"), indent=2)
r = j.get("result", {})
print(r.get("status"), r.get("winner"), r.get("winner_stats"), j.get("error"))
for a in r.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

In [ ]:
from google.colab import files
files.download("/content/quantize-as-a-tool/quantize_service.py")
files.download("/content/quantize-as-a-tool/memory.json")
files.download("/content/job_result.json")

In [ ]:
code = r'''import os
import requests
from mcp.server.fastmcp import FastMCP

URL = os.environ.get("QUANTIZE_URL", "").rstrip("/")
KEY = os.environ.get("QUANTIZE_API_KEY", "")

mcp = FastMCP("quantize-as-a-tool")


def _slim(d: dict) -> dict:
    keep = ["status", "model_id", "winner", "winner_stats", "fp16_size_gb",
            "budget_pct", "artifact_sha256", "artifact_dir", "flags"]
    out = {k: d[k] for k in keep if k in d}
    out["attempts"] = [
        {k: a[k] for k in ("config", "ppl_increase_pct", "size_gb", "verdict")}
        for a in d.get("attempts", [])
    ]
    return out


def _call(method, path, **kw):
    if not URL or not KEY:
        return None, {"status": "ERROR", "detail": "Set QUANTIZE_URL and QUANTIZE_API_KEY."}
    try:
        r = requests.request(method, URL + path, headers={"X-API-Key": KEY}, timeout=120, **kw)
    except requests.RequestException as e:
        return None, {"status": "ERROR", "detail": "Service unreachable: " + type(e).__name__}
    if r.status_code == 401:
        return None, {"status": "UNAUTHORIZED", "detail": "Wrong API key."}
    if r.status_code == 404:
        return None, {"status": "NOT_FOUND", "detail": "Unknown job id (the service may have restarted)."}
    if r.status_code != 200:
        return None, {"status": "ERROR", "detail": "HTTP " + str(r.status_code)}
    return r.json(), None


@mcp.tool()
def submit_quantize_job(model_id: str, accuracy_budget_pct: float = 10.0, bits: str = "auto") -> dict:
    """Start compressing a Hugging Face model (up to 3B params) to 4-bit or 8-bit.

    Returns a job_id immediately. The work can take several minutes for a new
    model, so call get_quantize_job(job_id) every 10-30 seconds until the status
    is DONE or FAILED. accuracy_budget_pct is the largest perplexity increase
    you accept. bits is "auto", "4" or "8".
    """
    data, err = _call("POST", "/jobs", json={"model_id": model_id,
                      "accuracy_budget_pct": accuracy_budget_pct, "bits": bits})
    return err or data


@mcp.tool()
def get_quantize_job(job_id: str) -> dict:
    """Check a job. Status is QUEUED, RUNNING, DONE or FAILED. When DONE, the
    result holds the chosen format, measured size, perplexity change and the
    artifact SHA-256, or a REFUSED status with flags explaining why."""
    data, err = _call("GET", "/jobs/" + job_id)
    if err:
        return err
    out = {"job_id": job_id, "status": data.get("status"), "model_id": data.get("model_id")}
    if data.get("result"):
        out["result"] = _slim(data["result"])
    if data.get("error"):
        out["error"] = data["error"]
    return out


@mcp.tool()
def quantize_model(model_id: str, accuracy_budget_pct: float = 10.0, bits: str = "auto") -> dict:
    """Synchronous version. Only use for models already measured (fast). For new
    models use submit_quantize_job, because long requests can time out."""
    data, err = _call("POST", "/quantize", json={"model_id": model_id,
                      "accuracy_budget_pct": accuracy_budget_pct, "bits": bits})
    return err or _slim(data)


@mcp.tool()
def service_health() -> dict:
    """Check whether the quantization service is reachable."""
    data, err = _call("GET", "/health")
    return err or data


if __name__ == "__main__":
    mcp.run()
'''
open("/content/mcp_server.py", "w").write(code)

# ---- test the new tools against the running service ----
import os, sys, time, json
os.environ["QUANTIZE_URL"] = open("/content/public_url.txt").read().strip()
os.environ["QUANTIZE_API_KEY"] = open("/content/api_key.txt").read().strip()
sys.path.insert(0, "/content")
sys.modules.pop("mcp_server", None)
import mcp_server

print("health:", mcp_server.service_health())
job = mcp_server.submit_quantize_job("TinyLlama/TinyLlama-1.1B-Chat-v1.0", 10.0)
print("submitted:", job)
for _ in range(60):
    j = mcp_server.get_quantize_job(job["job_id"])
    if j["status"] in ("DONE", "FAILED"):
        break
    time.sleep(5)
print(json.dumps(j, indent=2))
print("unknown job:", mcp_server.get_quantize_job("doesnotexist"))

from google.colab import files
files.download("/content/mcp_server.py")

In [ ]:
import time, requests, json
url = open("/content/public_url.txt").read().strip()
key = open("/content/api_key.txt").read().strip()
H = {"X-API-Key": key}

job = requests.post(url + "/jobs", headers=H, timeout=60,
                    json={"model_id": "HuggingFaceTB/SmolLM2-1.7B", "accuracy_budget_pct": 10.0}).json()
print("submitted:", job)

last = None
for _ in range(120):
    j = requests.get(f"{url}/jobs/{job['job_id']}", headers=H, timeout=60).json()
    if j["status"] != last:
        print(time.strftime("%H:%M:%S"), j["status"]); last = j["status"]
    if j["status"] in ("DONE", "FAILED"):
        break
    time.sleep(10)

json.dump(j, open("/content/job_result_smollm2.json", "w"), indent=2)
r = j.get("result", {})
print(r.get("status"), r.get("winner"), r.get("winner_stats"), r.get("flags"), j.get("error"))
for a in r.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

In [ ]:
import json
m = json.load(open("/content/quantize-as-a-tool/memory.json"))
e = m.get("HuggingFaceTB/SmolLM2-1.7B", {})
print("fp16:", e.get("fp16_ppl"), e.get("fp16_gb"))
for k, v in e.get("configs", {}).items():
    print(k, v)

In [ ]:
from google.colab import files
files.download("/content/quantize-as-a-tool/memory.json")
files.download("/content/job_result_smollm2.json")

In [ ]:
import time, json, base64, requests, os

# 1. patch quantize_service.py
p = "/content/quantize-as-a-tool/quantize_service.py"
src = open(p).read()
if "sign_report" not in src:
    new = '''
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives import serialization
import base64, datetime

KEY_PATH = os.environ.get("SIGNING_KEY_PATH", "/content/signing_key.pem")

def _load_signing_key():
    if os.path.exists(KEY_PATH):
        return serialization.load_pem_private_key(open(KEY_PATH, "rb").read(), password=None)
    k = Ed25519PrivateKey.generate()
    open(KEY_PATH, "wb").write(k.private_bytes(
        serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8, serialization.NoEncryption()))
    return k

_SIGN_KEY = _load_signing_key()
_PUB_B64 = base64.b64encode(_SIGN_KEY.public_key().public_bytes(
    serialization.Encoding.Raw, serialization.PublicFormat.Raw)).decode()

def _canon(d):
    return json.dumps(d, sort_keys=True, separators=(",", ":")).encode()

def sign_report(result):
    if result.get("status") not in ("SUCCESS", "NO_CONFIG_MET_BUDGET"):
        return result
    body = {k: v for k, v in result.items() if k != "signature"}
    body["signed_at"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    sig = _SIGN_KEY.sign(_canon(body))
    body["signature"] = {"alg": "ed25519", "public_key": _PUB_B64, "value": base64.b64encode(sig).decode()}
    return body

@app.get("/public-key")
def public_key():
    return {"alg": "ed25519", "public_key": _PUB_B64}

'''
    old = "        return quantize_smart(req.model_id, req.accuracy_budget_pct, bits, out_dir)"
    assert old in src
    src = src.replace(old, "        return sign_report(quantize_smart(req.model_id, req.accuracy_budget_pct, bits, out_dir))", 1)
    src = src.replace("def start_service(", new + "def start_service(", 1)
    open(p, "w").write(src)

# 2. standalone verifier (this file goes in the repo too)
VERIFY = '''import json, base64
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
from cryptography.exceptions import InvalidSignature

def verify_report(report, trusted_public_key_b64):
    """True only if the report is unmodified AND signed by the trusted key."""
    sig = report.get("signature") or {}
    if sig.get("public_key") != trusted_public_key_b64:
        return False
    body = {k: v for k, v in report.items() if k != "signature"}
    msg = json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
    try:
        Ed25519PublicKey.from_public_bytes(base64.b64decode(trusted_public_key_b64)).verify(
            base64.b64decode(sig["value"]), msg)
        return True
    except (InvalidSignature, KeyError, ValueError):
        return False
'''
open("/content/quantize-as-a-tool/verify_report.py", "w").write(VERIFY)
exec(VERIFY)

# 3. restart the service
try: server.should_exit = True
except Exception: pass
time.sleep(4)
get_ipython().run_line_magic("run", p)
server, key, url = start_service()
print("URL:", url)

# 4. signed result (local call, no tunnel timeout)
rep = requests.post("http://127.0.0.1:8000/quantize", headers={"X-API-Key": key}, timeout=3000,
                    json={"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "accuracy_budget_pct": 10.0}).json()
json.dump(rep, open("/content/signed_report_tinyllama.json", "w"), indent=2)
print(rep["status"], rep["winner"], rep["winner_stats"], rep["signed_at"])

# 5. verify like an outside agent: fetch the public key over the tunnel
pub = requests.get(url + "/public-key", timeout=60).json()["public_key"]
print("valid report     ->", verify_report(rep, pub))

bad = json.loads(json.dumps(rep)); bad["winner_stats"]["ppl_increase_pct"] = 1.0
print("tampered report  ->", verify_report(bad, pub))

from google.colab import files
for f in ["/content/quantize-as-a-tool/quantize_service.py",
          "/content/quantize-as-a-tool/verify_report.py",
          "/content/signed_report_tinyllama.json"]:
    files.download(f)

In [ ]:
import time, json, requests
key = open("/content/api_key.txt").read().strip()
url = open("/content/public_url.txt").read().strip()
H = {"X-API-Key": key}
MODEL = "allenai/OLMo-1B-hf"

job = requests.post(url + "/jobs", headers=H, timeout=60,
                    json={"model_id": MODEL, "accuracy_budget_pct": 10.0}).json()
print("submitted:", job)
last = None
for _ in range(120):
    j = requests.get(f"{url}/jobs/{job['job_id']}", headers=H, timeout=60).json()
    if j["status"] != last:
        print(time.strftime("%H:%M:%S"), j["status"]); last = j["status"]
    if j["status"] in ("DONE", "FAILED"):
        break
    time.sleep(10)

json.dump(j, open("/content/job_result_olmo.json", "w"), indent=2)
r = j.get("result", {})
print(r.get("status"), r.get("winner"), r.get("winner_stats"), r.get("flags"), j.get("error"))
for a in r.get("attempts", []):
    print("  ", a["config"], a["ppl_increase_pct"], a["size_gb"], a["verdict"], a["source"])

In [ ]:
from huggingface_hub import logout
try: logout()
except Exception: pass

import os
print("token file left:", os.path.exists(os.path.expanduser("~/.cache/huggingface/token")))

from google.colab import files
files.download("/content/quantize-as-a-tool/memory.json")
files.download("/content/job_result_olmo.json")

In [ ]:
import json, math, time, warnings, logging, torch
warnings.filterwarnings("ignore"); logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

N = 400
ARC = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test").select(range(N))

@torch.no_grad()
def arc_accuracy(model, tok):
    model.eval(); correct = 0
    for ex in ARC:
        ctx = f"Question: {ex['question']}\nAnswer:"
        ctx_len = len(tok(ctx).input_ids)
        scores = []
        for choice in ex["choices"]["text"]:
            ids = tok(ctx + " " + choice, return_tensors="pt").input_ids.to("cuda")
            logp = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
            tgt = ids[0, 1:]
            ll = logp[torch.arange(len(tgt)), tgt][ctx_len - 1:].sum().item()
            scores.append(ll / max(len(choice), 1))        # length-normalised
        pred = ex["choices"]["label"][scores.index(max(scores))]
        correct += pred == ex["answerKey"]
    return round(100 * correct / N, 1)

ladder = dict(build_ladder())
plan = {"Qwen/Qwen2.5-0.5B": ["fp16", "nf4+double_quant", "int8"],
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0": ["fp16", "nf4+double_quant"]}
results = {}
for mid, cfgs in plan.items():
    tok = AutoTokenizer.from_pretrained(mid)
    for c in cfgs:
        t = time.time()
        kw = {"dtype": torch.float16} if c == "fp16" else {"quantization_config": ladder[c]}
        m = AutoModelForCausalLM.from_pretrained(mid, device_map="cuda", **kw)
        acc = arc_accuracy(m, tok)
        results.setdefault(mid, {})[c] = acc
        print(f"{mid} | {c}: ARC-Easy acc = {acc}%  ({time.time()-t:.0f}s)")
        del m; free_gpu()

json.dump({"task": "ARC-Easy", "n": N, "results": results}, open("/content/benchmarks_arc.json", "w"), indent=2)
print(open("/content/benchmarks_arc.json").read())

In [ ]:
from google.colab import files
files.download("/content/benchmarks_arc.json")

In [ ]:
import subprocess, re
want = "torch|transformers|bitsandbytes|accelerate|huggingface-hub|safetensors|datasets|fastapi|uvicorn|pydantic|starlette|cryptography"
out = subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout
lines = [l for l in out.splitlines() if re.match(rf"^({want})==", l, re.I)]
open("/content/requirements-lock.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
from google.colab import files
files.download("/content/requirements-lock.txt")